In [ ]:
import torch
from VGG import VGG16
from data import Dataset
from model_utils import ModelManager

In [ ]:
# Initialize dataset and load test data
dataset = Dataset(batch_size=32, num_workers=10)
testloader = dataset.testloader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'Using device: {device}')
print(f'Test set size: {len(dataset.testset)} samples')

  3%|▎         | 5.24M/170M [00:10<05:41, 485kB/s] 


KeyboardInterrupt: 

In [ ]:
# Load saved model
print('Loading saved model...\n')
manager = ModelManager()

# Create fresh model instance for loading
loaded_model = VGG16(num_classes=10)
loaded_optimizer = loaded_model.optimizer

# Load checkpoint
load_result = manager.load_model(loaded_model, loaded_optimizer)
if load_result:
    epoch_loaded, saved_results = load_result
    print(f'Successfully loaded model from epoch {epoch_loaded}\n')
else:
    print('Failed to load model. Make sure to run training first.')
    saved_results = {}

In [ ]:
# Display model architecture
manager.get_model_info(loaded_model)

In [ ]:
# Test the loaded model
print('Testing loaded model on test set:\n')
test_accuracy = manager.test_model(loaded_model, testloader, loaded_model.device)

In [ ]:
# Display final comparison
print('\n' + '='*60)
print('TEST RESULTS')
print('='*60)
if saved_results and 'val_accuracies' in saved_results:
    print(f'Best Val Accuracy (during training): {max(saved_results.get("val_accuracies", [0])):.2f}%')
    print(f'Final Train Accuracy: {saved_results["train_accuracies"][-1]:.2f}%')
print(f'Test Accuracy (now): {test_accuracy:.2f}%')
print('='*60)

In [ ]:
# Test on custom images from test_images folder
import os
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# CIFAR-10 class names
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Prepare transforms (same as training)
normalize = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
test_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize
])

# Load all images from test_images folder
test_images_dir = './test_images'
test_images_list = sorted([f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))])

print(f'Found {len(test_images_list)} test images:')
for img in test_images_list:
    print(f'  - {img}')

In [ ]:
# Function to run inference on a single image
def predict_image(model, image_path, device, transform):
    """
    Predict class for a single image.
    Returns: (predicted_class, confidence, all_probs)
    """
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)  # Add batch dimension
    
    # Forward pass
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)
    
    return predicted_idx.item(), confidence.item(), probabilities[0].cpu()

# Run inference on all test images
print('='*70)
print('TESTING ON CUSTOM IMAGES')
print('='*70 + '\n')

results = []
for img_file in test_images_list:
    img_path = os.path.join(test_images_dir, img_file)
    pred_idx, confidence, all_probs = predict_image(loaded_model, img_path, loaded_model.device, test_transform)
    pred_class = CIFAR10_CLASSES[pred_idx]
    results.append({
        'filename': img_file,
        'predicted_class': pred_class,
        'confidence': confidence,
        'all_probs': all_probs
    })
    print(f'Image: {img_file:<20} | Predicted: {pred_class:<12} | Confidence: {confidence*100:.2f}%')

print('\n' + '='*70)

In [ ]:
# Visualize predictions with matplotlib
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

for idx, result in enumerate(results):
    ax = axes[idx]
    
    # Load and display image
    img_path = os.path.join(test_images_dir, result['filename'])
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    
    # Format title with prediction and confidence
    pred_class = result['predicted_class']
    confidence = result['confidence']
    title = f'Pred: {pred_class}\nConfidence: {confidence*100:.1f}%'
    ax.set_title(title, fontsize=11, fontweight='bold', color='green' if confidence > 0.8 else 'orange')
    ax.axis('off')

plt.suptitle('VGG16 CIFAR-10 Predictions on Test Images', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print('Visualization complete!')

In [ ]:
# Detailed confidence analysis
print('\n' + '='*70)
print('DETAILED PREDICTION ANALYSIS')
print('='*70 + '\n')

confidences = [r['confidence'] for r in results]
avg_confidence = sum(confidences) / len(confidences)

print(f'Average Confidence: {avg_confidence*100:.2f}%')
print(f'Min Confidence: {min(confidences)*100:.2f}%')
print(f'Max Confidence: {max(confidences)*100:.2f}%')
print(f'\nConfidence Levels:')
print(f'  - Very High (>90%): {sum(1 for c in confidences if c > 0.9)} images')
print(f'  - High (80-90%): {sum(1 for c in confidences if 0.8 <= c <= 0.9)} images')
print(f'  - Medium (70-80%): {sum(1 for c in confidences if 0.7 <= c < 0.8)} images')
print(f'  - Low (<70%): {sum(1 for c in confidences if c < 0.7)} images')

print(f'\n{"Image File":<20} {"Predicted":<15} {"Confidence":<15} {"Top 3 Predictions":<30}')
print('-'*80)

for result in results:
    img_file = result['filename']
    pred_class = result['predicted_class']
    confidence = result['confidence']
    
    # Get top 3 predictions
    top3_probs, top3_idx = torch.topk(result['all_probs'], 3)
    top3_str = ', '.join([f"{CIFAR10_CLASSES[idx]}: {prob*100:.1f}%" 
                          for prob, idx in zip(top3_probs, top3_idx)])
    
    print(f'{img_file:<20} {pred_class:<15} {confidence*100:>6.2f}%        {top3_str:<30}')

print('='*80)

In [ ]:
# Test a single custom image (interactive)
def test_single_image(image_path):
    """
    Test a single image and display prediction with confidence.
    """
    if not os.path.exists(image_path):
        print(f'Error: Image not found at {image_path}')
        return
    
    # Run prediction
    pred_idx, confidence, all_probs = predict_image(loaded_model, image_path, loaded_model.device, test_transform)
    pred_class = CIFAR10_CLASSES[pred_idx]
    
    # Get top 5 predictions
    top5_probs, top5_idx = torch.topk(all_probs, 5)
    
    # Display image and prediction
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Show image
    img = Image.open(image_path).convert('RGB')
    ax1.imshow(img)
    ax1.set_title(f'Predicted: {pred_class}\nConfidence: {confidence*100:.2f}%', 
                  fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # Show confidence bar chart
    ax2.barh(range(5), top5_probs.numpy(), color='steelblue')
    ax2.set_yticks(range(5))
    ax2.set_yticklabels([CIFAR10_CLASSES[idx] for idx in top5_idx])
    ax2.set_xlabel('Confidence Score')
    ax2.set_title('Top 5 Predictions')
    ax2.set_xlim([0, 1])
    
    # Add percentage labels on bars
    for i, (prob, idx) in enumerate(zip(top5_probs, top5_idx)):
        ax2.text(prob + 0.01, i, f'{prob*100:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\n✓ Prediction: {pred_class}')
    print(f'✓ Confidence: {confidence*100:.2f}%')
    print(f'\nTop 5 Predictions:')
    for i, (prob, idx) in enumerate(zip(top5_probs, top5_idx), 1):
        print(f'  {i}. {CIFAR10_CLASSES[idx]:<12} - {prob*100:6.2f}%')

# Example usage - uncomment to test a custom image:
# test_single_image('./test_images/cat.jpg')